# 동아대 대학원 QML 강의 - 2일차

## Colab 실습: 환경 구성 → EDA → 전처리 → 회로 인코딩 → QGAN 학습

**이번 시간의 목표**: HAM10000 실데이터로 전처리 파이프라인을 직접 돌리고,
**QGAN(양자 생성자 + 고전 판별자)을 라이브로 학습시켜 피부병변 이미지를 생성함.**

1. HAM10000 다운로드 + EDA + 세그멘테이션 전처리
2. 서브제너레이터(변분 양자회로) 구성 + 출력 확인
3. QGAN 학습 → 생성 → 생성품질 지표 4종으로 평가

**시간 배분**:
환경세팅 → EDA → 전처리 실습 → 회로 기초 → QGAN 학습 → 생성·평가 → 정리

**오늘 쓰는 회로 규모**: 데이터 6 + 보조 1 = 7큐빗, patch 64, 서브제너레이터 4개, depth 10.
depth는 논문과 같고, 서브제너레이터 개수와 큐빗 수는 Colab 시뮬레이터에서 수업 시간 안에
학습이 끝나도록 줄임(논문은 서브제너레이터 16개 x 데이터 7 + 보조 1큐빗).

## 0. Google Colab 시작하기 (수업 전 준비 권장)

이 실습은 모두 **Google Colab**에서 진행함. Colab은 웹 브라우저에서 Python 노트북을 실행하는 Google의 무료 클라우드 서비스임.
- 개인 PC에 Python·라이브러리를 설치할 필요가 없음.
- 필요한 것: **Google 계정 + 크롬 등 최신 브라우저**. 노트북 PC 사양·운영체제(Windows/Mac)와 무관함.
- 메뉴 이름은 Google 계정 언어 설정에 따라 한국어/영어로 표시됨.

### 0-1. Google 계정 만들기 (이미 있으면 건너뜀)
1. https://accounts.google.com/signup 접속
2. 이름 → 생년월일·성별 → 사용할 Gmail 주소 → 비밀번호 순으로 입력
3. 휴대전화 인증(요청 시) 후 약관 동의 → 생성 완료
4. https://colab.research.google.com 에 접속해 새로 만든 계정으로 로그인되는지 확인

> 학교·회사 계정(Google Workspace)은 관리자가 Colab 사용을 막아둔 경우가 있음. 접속이 안 되면 **개인 Gmail 계정**을 사용함.

### 0-2. 실습 노트북 열기
- **Repository 실행**: https://github.com/hyunhp/skin_disease_qml 첫 화면의 **Open in Colab** 링크 클릭 → Colab에서 바로 열림
- 연 직후 반드시 **`파일(File) → Drive에 사본 저장(Save a copy in Drive)`** 실행 - 사본을 만들지 않으면 수정·실행 결과가 본인 Drive에 저장되지 않음.
- 처음 실행할 때 "이 노트북은 Google에서 작성하지 않았습니다(This notebook was not authored by Google)" 경고가 뜨면 `무시하고 계속(Run anyway)`을 누름 

### 0-3. 기본 사용법
| 하고 싶은 것 | 방법 |
|---|---|
| 셀 실행 | 셀을 클릭하고 **Shift + Enter**, 또는 셀 왼쪽 ▶ 버튼 |
| 실행 중인지 확인 | 셀 왼쪽 버튼이 회전 중이면 실행 중, 끝나면 `[숫자]`로 바뀜 |
| 실행 중단 | 셀 왼쪽 ■ 버튼, 또는 `런타임(Runtime) → 실행 중단(Interrupt execution)` |
| 처음부터 전부 실행 | `런타임(Runtime) → 모두 실행(Run all)` |
| 꼬였을 때 초기화 | `런타임(Runtime) → 세션 다시 시작(Restart session)` 후 위에서부터 다시 실행 |
| 파일 확인 | 왼쪽 사이드바 📁(파일) 아이콘 |

- 코드 셀은 **반드시 위에서부터 순서대로** 실행함. 앞 셀을 건너뛰면 `NameError`(변수·함수가 정의되지 않음)가 발생함.
- 이 실습은 **기본 CPU 런타임으로 충분**함. `런타임 유형 변경`에서 GPU로 바꿀 필요 없음.
- Google Drive 연결(5번 섹션) 시 권한 요청 팝업이 뜨면 본인 계정을 선택하고 허용함.

### 0-4. 비고
- Colab의 가상 머신(런타임)은 **일정 시간 사용하지 않거나 세션을 오래 유지하면 자동으로 끊김**(무료 버전 기준 대략 유휴 90분, 최대 12시간).
- 끊기면 설치한 라이브러리·내려받은 데이터·변수가 **모두 사라짐** → 다시 연결한 뒤 1번 섹션부터 다시 실행하면 됨(노트북 파일과 Drive에 저장한 파일은 남음).
- 그래서 오늘 결과물(전처리 데이터)은 마지막 5번 섹션에서 **Google Drive에 저장**함.
- 오류가 나면 당황하지 말고 **오류 메시지 마지막 줄**을 확인함 - 대부분 앞 셀 미실행 또는 연결 끊김이 원인임.

## 1. 환경 세팅

아래 셀을 순서대로 실행함. 전부 실행되면 다음 섹션으로 진행함.

In [ ]:
# 필요 라이브러리 설치 
!pip install -q pennylane pennylane-lightning scikit-fuzzy kagglehub koreanize-matplotlib
print("설치 완료")

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import pennylane as qml
import skfuzzy as fuzz
from sklearn.cluster import KMeans

try:
    import koreanize_matplotlib  # 나눔고딕 폰트 등록 + 기본 폰트 지정
except Exception as e:
    print("한글 폰트 설정 실패 - 그래프의 한글만 깨지고 실습 진행에는 지장 없음:", e)

print(f"torch: {torch.__version__}")
print(f"pennylane: {qml.__version__}")
print(f"opencv: {cv2.__version__}")

# 간단한 동작 확인 - 1큐빗 회로가 정상 실행되는지
dev = qml.device("lightning.qubit", wires=1)
@qml.qnode(dev)
def _sanity_check():
    qml.Hadamard(wires=0)
    return qml.expval(qml.PauliZ(0))

print(f"PennyLane 동작 확인: {_sanity_check()}")

## 2. HAM10000 데이터셋 로드 + EDA

HAM10000("Human Against Machine with 10000 training images")은 10,015장의 피부경검사(dermatoscopic)
이미지로 구성된 피부병변 데이터셋임. 

### 데이터 다운로드 방식: kagglehub
- `kagglehub` 라이브러리로 Kaggle 데이터셋을 Colab 런타임에 직접 내려받음. (`kaggle.json` 업로드 및 수동 압축 해제 과정 불필요)
- 데이터셋 전체 용량은 **약 6GB**임. 개인 PC가 아닌 Colab 런타임(클라우드 디스크)에만 저장되므로 PC 용량 부담 없음.
- 다운로드 파일은 kagglehub 캐시 폴더(`~/.cache/kagglehub`)에 저장됨. 같은 런타임에서 셀을 재실행하면 캐시를 재사용하므로 재다운로드하지 않음. 런타임이 초기화되면 캐시도 삭제되어 재다운로드 필요함.
- Colab에서는 kagglehub가 Kaggle의 Colab 전용 캐시에서 데이터셋을 바로 연결(mount)하는 경우도 있음. 이때는 경로가 `/kaggle/input/...`으로 표시되고 수 초 만에 끝남(정상 - 이후 코드는 동일하게 동작함).
- 원본 6GB는 Drive에 복사하지 않음. **전처리 결과(`.npy`, 소용량)만 Google Drive에 저장**하여 3일차에 사용함(5번 섹션).

### Kaggle 로그인 - 기본적으로 불필요함
HAM10000은 **공개 데이터셋이라 Kaggle 계정·API 토큰 없이 바로 다운로드 가능함.** 아래 (선택) 로그인 셀은 건너뛰고 다운로드 셀부터 실행함.
다운로드 셀에서 인증 오류(`UnauthenticatedError`, 401/403)가 발생한 경우에만 로그인을 진행함:
1. Kaggle 계정으로 로그인한 뒤 https://www.kaggle.com/settings 접속 → API 항목에서 새 토큰 발급(Create New Token) 후 복사
2. 로그인 셀의 `# kagglehub.login()` 주석을 해제하고 실행 → 입력창에 토큰 붙여넣기 → **Login 버튼 클릭**
3. "Kaggle credentials successfully validated" 메시지를 **확인한 뒤에** 다운로드 셀을 다시 실행함 (로그인 셀은 입력을 기다리지 않고 바로 종료되므로, 메시지 확인 전에 다음 셀을 실행하면 인증 없이 진행됨)
4. 토큰은 비밀번호와 동일하게 취급하며, 노트북에 직접 기재하거나 타인에게 공유하지 않음.

In [ ]:
import kagglehub

# # (선택) 다운로드 셀에서 인증 오류가 발생한 경우에만 아래 주석을 해제하고 실행함 - 공개 데이터셋이라 보통은 불필요
# # 입력창에 토큰 붙여넣기 → Login 클릭 → 성공 메시지 확인 후 다운로드 셀 재실행
# kagglehub.login()

# # (대안) 환경변수로 직접 지정하는 방식
# os.environ["KAGGLE_USERNAME"] = "본인 Kaggle 아이디"
# os.environ["KAGGLE_KEY"] = "본인 API key"

In [ ]:
import kagglehub

# 최신 버전 다운로드 (이미 캐시에 있으면 재사용) - 약 6GB
# Colab 전용 캐시에서 바로 연결되는 경우 수 초 만에 끝나고 경로가 /kaggle/input/... 으로 표시됨
try:
    DATA_DIR = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
except Exception as e:
    raise RuntimeError("다운로드 실패 - 네트워크 상태 확인 후 재실행하고, 인증 오류라면 위 (선택) 로그인 셀을 "
                       "실행한 뒤 다시 시도할 것") from e
print("데이터셋 경로:", DATA_DIR)
print("포함 항목:", sorted(os.listdir(DATA_DIR)))

#### 전체 흐름 한눈에 보기 (EDA·전처리 파이프라인 개요)

```
원본 이미지 (600×450 RGB, 10,015장)
   │ ① EDA: 클래스 분포·샘플 확인 → DF vs NV 이진분류로 범위 축소
   ▼
DF 115장 / NV 6,705장 (약 1:58 불균형)
   │ ② 세그멘테이션: grayscale → 64×64 축소 → K-means → Fuzzy C-means → 병변만 남김
   ▼
병변만 남은 64×64 grayscale
   │ ③ 초저해상도 축소: 64×64 → 16×16 (0~1 정규화)
   ▼
16×16 = 256픽셀 벡터
   │ ④ 양자 인코딩: 256 = 서브제너레이터 4개 × 64픽셀(2⁶) → 각 회로는 데이터 큐빗 6 + 보조 1 = 7큐빗
   ▼
QGAN 학습 (오늘 라이브로 진행) → 생성 → 생성품질 지표 4종으로 평가
```

#### 단계별 핵심

**① EDA (2번 섹션)**
- HAM10000은 7개 클래스로 구성되나 극단적으로 불균형함 - NV(모반)가 전체의 약 67%, DF(피부섬유종)는 약 1.1%에 불과함.
- 논문(Andra et al.)과 동일하게 **최소 클래스 DF vs 최다 클래스 NV** 조합을 선택함. 불균형이 가장 심한 조합이라 "생성 모델로 소수 클래스를 증강하면 분류가 나아지는가"라는 질문을 검증하기에 적합함.
- 확인할 것: 클래스별 이미지 수(막대그래프), 클래스별 샘플 이미지의 시각적 차이.

**② 세그멘테이션 (3번 섹션) - 파이프라인의 핵심 단계**
- 논문 Table 10에서 QGAN 증강은 **원본(raw) 이미지에서는 오히려 성능을 떨어뜨렸고, 세그멘테이션을 거친 이미지에서만 개선됨.**
- 이유: 큐빗 수 제약으로 이미지를 16×16까지 줄여야 함. 배경(피부·털·조명 얼룩)이 섞인 채로 줄이면 256픽셀 안에 병변 정보가 거의 남지 않음. 병변만 남겨두고 줄여야 적은 픽셀로도 "병변의 모양·위치" 정보가 보존됨.
- 방법: 밝기값만으로 픽셀을 4개 그룹으로 나눔 → K-means로 대략 나눈 결과를 출발점으로 Fuzzy C-means(각 픽셀이 여러 그룹에 부분적으로 속하는 방식)가 경계를 정제함 → **가장 어두운 그룹 = 병변**으로 보고 나머지는 0(검정)으로 지움.
- 한계(직접 확인해볼 것): 피부경 이미지 모서리의 검은 테두리(비네팅)가 병변보다 어두우면 테두리가 병변으로 잘못 선택될 수 있음. 실제 결과 이미지에서 이런 사례가 보이는지 관찰함.

**③ 초저해상도 축소 (3번 섹션 후반)**
- 64×64 → 16×16으로 축소하고 0~1로 정규화함. 사람이 보기엔 형체만 남지만, 이 수준이 현재 시뮬레이터로 다룰 수 있는 규모임.
- `IMG_SIZE`를 8로 바꾸면 8×8=64픽셀(서브제너레이터 4개 × 16픽셀 → 데이터 큐빗 4개)로 더 작게 실습 가능함. 단 3일차 자산과 해상도가 달라지므로 비교는 안 됨.

**④ 양자 인코딩 + 회로 (4번 섹션)**
- 회로 하나가 256픽셀을 전부 만들려면 데이터 큐빗 8개가 필요해 비용이 커짐. 그래서 **서브제너레이터 4개가 64픽셀씩(= 2⁶, 데이터 큐빗 6개) 나눠 만들고 이어붙임**(patch 방식, 논문 Table 3).
- 각 회로는 데이터 큐빗 6 + 보조 큐빗 1 = **7큐빗**이며, 보조큐빗이 0인 경우의 확률 64개를 patch 한 장으로 씀. 큐빗 수가 1개 늘 때마다 표현 가능한 픽셀 수가 2배가 됨.
- 오늘은 회로 출력을 먼저 확인한 뒤(학습 전이라 랜덤 패턴), **5번 섹션에서 실제로 학습시켜** 생성 결과를 지표로 평가함.

#### 생각해볼 질문 (다운로드를 기다리며)
1. 불균형 1:58 상태로 분류기를 학습하면 어떤 문제가 생기는가? (힌트: 전부 NV라고 답해도 정확도 98%)
2. 16×16까지 줄인 이미지로도 DF와 NV를 구분할 수 있을까? 어떤 정보가 남고 어떤 정보가 사라지는가?
3. 세그멘테이션을 하지 않고 원본을 바로 16×16으로 줄이면 어떤 이미지가 될지 예상해봄.
4. 회로 1개로 256픽셀을 만드는 것과, 작은 회로 4개가 64픽셀씩 나눠 만드는 것은 무엇이 다른가?

In [ ]:
metadata = pd.read_csv(f"{DATA_DIR}/HAM10000_metadata.csv")
print(f"전체 이미지 수: {len(metadata)}")
metadata.head()

In [ ]:
# 클래스별 분포 (EDA) - 논문 Fig.12와 동일한 그래프
class_names_kr = {"akiec": "광선각화증", "bcc": "기저세포암", "bkl": "양성각화증",
                   "df": "피부섬유종", "mel": "흑색종", "nv": "모반(정상)", "vasc": "혈관병변"}
counts = metadata["dx"].value_counts()
plt.figure(figsize=(8, 4))
plt.bar([class_names_kr[c] for c in counts.index], counts.values)
plt.ylabel("이미지 수"); plt.title("HAM10000 클래스별 분포")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
print(counts)
print(f"\nDF(피부섬유종): {counts.get('df', 0)}장 vs NV(모반): {counts.get('nv', 0)}장",
      f"- 불균형 비율 약 1:{counts.get('nv',0)//max(1,counts.get('df',1))}")

In [ ]:
# 샘플 이미지 확인
def find_image_path(image_id):
    for sub in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        p = f"{DATA_DIR}/{sub}/{image_id}.jpg"
        if os.path.exists(p):
            return p
    return None

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, cls in zip(axes.flat, class_names_kr.keys()):
    row = metadata[metadata["dx"] == cls].iloc[0]
    path = find_image_path(row["image_id"])
    if path:
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.set_title(f"{cls} ({class_names_kr[cls]})"); ax.axis("off")
axes.flat[-1].axis("off")  # 7개 클래스 → 8칸 중 마지막 칸은 비움
plt.tight_layout(); plt.show()

## 3. 전처리 실습 - 세그멘테이션 (30분)

**세그멘테이션이 중요한 이유**: 논문 실험(Table 10)에서 QGAN 증강은 raw 이미지에서는 오히려 성능을
하락시켰고, 세그멘테이션 전처리를 거친 이미지에서만 성능이 개선됨. 따라서 이 단계가 파이프라인의 핵심임.

논문 Section 3.1 방식: grayscale 변환 → K-means 대략적 클러스터링 → Fuzzy C-means 정제
→ 병변(가장 어두운 클러스터)만 남기고 배경 제거.

In [ ]:
def segment_fcm_kmeans(img_bgr, n_clusters=4, resize_to=None):
    """HAM10000 QGAN 논문 Section 3.1 - K-means+Fuzzy C-means 하이브리드 세그멘테이션."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if resize_to is not None:
        gray = cv2.resize(gray, resize_to, interpolation=cv2.INTER_AREA)
    h, w = gray.shape
    pixels = gray.reshape(-1, 1).astype(np.float32)

    # 1단계: K-means로 대략적 클러스터링 (논문: "pre-clustering으로 FCM 처리시간 단축")
    km = KMeans(n_clusters=n_clusters, n_init=4, random_state=0).fit(pixels)
    # K-means 중심까지의 거리로 FCM 초기 소속도 행렬 구성 (m=2 → 소속도 ∝ 1/거리²)
    dist = np.abs(pixels - km.cluster_centers_.reshape(1, -1)) + 1e-6
    u_init = 1.0 / dist ** 2
    u_init = (u_init / u_init.sum(axis=1, keepdims=True)).T  # (클러스터 수, 픽셀 수)

    # 2단계: Fuzzy C-means로 정제 (K-means 결과에서 출발 → 무작위 시작보다 빨리 수렴, 매번 같은 결과)
    cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
        pixels.T, c=n_clusters, m=2.0, error=1e-4, maxiter=100, init=u_init)
    membership = np.argmax(u, axis=0)
    lesion_cluster = np.argmin(cntr.flatten())  # 가장 어두운 클러스터 = 병변
    mask = (membership == lesion_cluster).reshape(h, w).astype(np.uint8) * 255
    return cv2.bitwise_and(gray, gray, mask=mask)

print("함수 정의 완료")

In [ ]:
# DF 클래스 샘플 이미지로 세그멘테이션 테스트
df_row = metadata[metadata["dx"] == "df"].iloc[0]
img_bgr = cv2.imread(find_image_path(df_row["image_id"]))
segmented = segment_fcm_kmeans(img_bgr, n_clusters=4, resize_to=(64, 64))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title("원본"); axes[0].axis("off")
axes[1].imshow(segmented, cmap="gray"); axes[1].set_title("세그멘테이션 결과"); axes[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 큐빗 인코딩용 초저해상도 축소 - 16x16=256픽셀을 서브제너레이터 4개가 64픽셀씩 나눠 생성
IMG_SIZE = 16        # 2026 재설계 확정 스펙(3일차와 동일해야 함)
N_GENERATORS = 4     # 서브제너레이터 개수 - 각자 patch 64픽셀 담당

def resize_for_encoding(img_gray, size):
    return cv2.resize(img_gray, size, interpolation=cv2.INTER_AREA)

small = resize_for_encoding(segmented, (IMG_SIZE, IMG_SIZE))
plt.figure(figsize=(3, 3))
plt.imshow(small, cmap="gray"); plt.title(f"{IMG_SIZE}x{IMG_SIZE} 축소본(양자 인코딩용)"); plt.axis("off")
plt.show()

patch_px = IMG_SIZE * IMG_SIZE // N_GENERATORS
n_data_q = int(np.log2(patch_px))
print(f"전체 픽셀 수: {IMG_SIZE*IMG_SIZE}")
print(f"서브제너레이터 {N_GENERATORS}개 -> patch당 {patch_px}픽셀 -> 데이터 큐빗 {n_data_q}개 (2^{n_data_q}={patch_px})")
print("참고: 논문은 서브제너레이터 16개 x (데이터 7 + 보조 1)큐빗이며, 여기서는 수업 시간에 맞춰 줄인 것임")

In [ ]:
# 전처리 함수 - 파이프라인을 직접 돌려보는 용도
def preprocess_class(metadata, dx_class, img_size, max_n=None):
    rows = metadata[metadata["dx"] == dx_class]
    if max_n:
        rows = rows.head(max_n)
    imgs = []
    for _, row in rows.iterrows():
        path = find_image_path(row["image_id"])
        if path is None:
            continue
        bgr = cv2.imread(path)
        seg = segment_fcm_kmeans(bgr, n_clusters=4, resize_to=(64, 64))
        imgs.append(resize_for_encoding(seg, (img_size, img_size)).astype(np.float32) / 255.0)
    return np.stack(imgs) if imgs else np.zeros((0, img_size, img_size), dtype=np.float32)

# 수업 중에는 소량만 직접 전처리함(1장당 수 초 소요). 학습에 쓸 DF 115장 전체는 다음 섹션에서
# 강의 저장소의 전처리 완료 자산을 받아 사용함 - 전원이 동일한 데이터로 학습해야 결과 비교가 됨
df_demo = preprocess_class(metadata, "df", IMG_SIZE, max_n=5)
nv_demo = preprocess_class(metadata, "nv", IMG_SIZE, max_n=5)
print(f"직접 전처리한 샘플 - DF {df_demo.shape}, NV {nv_demo.shape}")

fig, axes = plt.subplots(2, 5, figsize=(11, 4.6))
for j in range(5):
    axes[0, j].imshow(df_demo[j], cmap="gray"); axes[0, j].set_title(f"DF {j}", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(nv_demo[j], cmap="gray"); axes[1, j].set_title(f"NV {j}", fontsize=9); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

## 4. 인코딩 + 회로 구성

이미지를 양자 회로에 입력할 준비가 완료됨. HAM10000 QGAN 논문의 서브제너레이터 하나를
PennyLane으로 직접 구성함. 먼저 **학습 전 회로 출력**을 확인하고, 다음 섹션에서 같은 회로를 실제로 학습시킴.

구조: ① 노이즈를 Ry 회전각으로 인코딩 → ② 학습 가능한 변분 레이어(Ry 회전 + CZ 얽힘) 반복
→ ③ 전체 큐빗을 측정한 뒤 **보조큐빗이 0인 경우의 확률만** 이미지 값으로 사용

**보조큐빗(ancilla)을 두는 이유**: 데이터 큐빗만 측정하면 64개 확률의 합이 항상 1로 고정됨. 보조큐빗이 0인
경우만 골라 쓰면(논문의 사후선택 post-selection) 합이 0~1 사이에서 달라질 수 있어, 여러 회로가
이미지 조각(patch)을 나눠 만들 때 "배경이라 어두운 조각"과 "병변이 있어 밝은 조각"을 구분해 표현할 수 있음.

In [ ]:
import math

N_DATA_QUBITS = int(np.log2(IMG_SIZE * IMG_SIZE // N_GENERATORS))  # 256/4=64 -> 6큐빗
N_ANCILLA = 1
N_QUBITS = N_DATA_QUBITS + N_ANCILLA
Q_DEPTH = 10  # 논문과 동일

dev = qml.device("lightning.qubit", wires=N_QUBITS)

@qml.qnode(dev)
def sub_generator_circuit(noise, weights):
    # 1) 노이즈 angle encoding
    for i in range(N_QUBITS):
        qml.RY(noise[i], wires=i)
    # 2) 변분 레이어
    for layer in range(Q_DEPTH):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i], wires=i)
        for i in range(N_QUBITS - 1):
            qml.CZ(wires=[i, i + 1])
    # 3) 보조큐빗까지 전체 측정 (출력 2^7=128개 확률)
    return qml.probs(wires=range(N_QUBITS))

print(f"큐빗 수: {N_QUBITS} (데이터 {N_DATA_QUBITS} + 보조 {N_ANCILLA}), depth={Q_DEPTH}")

# 랜덤 파라미터로 forward pass만 확인 (아직 학습 전)
noise = np.random.rand(N_QUBITS) * math.pi
weights = np.random.rand(Q_DEPTH, N_QUBITS) * math.pi
probs_all = sub_generator_circuit(noise, weights)
# 보조큐빗(마지막 큐빗)이 0인 경우의 확률만 선택 = 짝수 번째 항목 -> 64개 patch (논문의 사후선택에 해당)
output = probs_all[::2]
print(f"전체 측정 확률 개수: {probs_all.shape} (=2^{N_QUBITS}) / 전체 합: {float(np.sum(probs_all)):.6f} (1이어야 정상)")
print(f"보조큐빗=0 항목만 선택한 출력 shape: {output.shape} (=2^{N_DATA_QUBITS}={IMG_SIZE*IMG_SIZE//N_GENERATORS}이어야 함)")
print(f"선택된 확률의 합 (0~1 사이 - 이 값도 학습으로 조절되는 '밝기 총합'임): {float(np.sum(output)):.4f}")

In [ ]:
# 서브제너레이터 1개의 출력은 이미지 전체가 아니라 patch(절반)임 - 학습 전이라 랜덤한 패턴이 정상
patch_side = int(np.sqrt(len(output) * N_GENERATORS))  # 16
plt.figure(figsize=(3, 2))
plt.imshow(np.array(output).reshape(-1, patch_side), cmap="gray")
plt.title("학습 전 서브제너레이터 출력 patch (랜덤 - 정상)")
plt.axis("off")
plt.show()
print(f"patch 1개 = {len(output)}픽셀 = {IMG_SIZE}x{IMG_SIZE} 이미지의 1/{N_GENERATORS}")
print(f"다음 섹션에서 이 회로 {N_GENERATORS}개를 묶어 학습시키면 병변 이미지 패턴을 생성하게 됨")

## 5. QGAN 학습 (라이브)

지금까지 만든 서브제너레이터 4개를 묶어 **생성자**로 쓰고, 논문 Table 4의 **고전 판별자**(5층 완전연결)와
경쟁시켜 학습함. 학습이 끝나면 생성자가 DF(피부섬유종) 세그멘테이션 이미지를 닮은 패턴을 만들어냄.

**데이터**: 전원이 같은 조건으로 학습하도록 강의 저장소의 전처리 완료 자산(DF 115장)을 받아 씀.
직접 전처리한 `df_demo`로 바꿔 돌려도 되지만, 5장으로는 학습이 되지 않음.

**척도 주의**: 생성자 출력은 확률(patch별 합 0~1)이므로, 실제 이미지도 **픽셀 합 = 1**로 맞춰서 비교함.
눈으로 보거나 지표를 계산할 때는 0~1로 되돌림(`to_unit_scale(x, "prob")`).

In [ ]:
# 강의 저장소 내려받기 (전처리 자산 + 지표 모듈 사용)
import sys, subprocess, os

REPO_DIR = "/content/skin_disease_qml"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/hyunhp/skin_disease_qml.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

ASSET_DIR = f"{REPO_DIR}/day3/assets"
sys.path.insert(0, f"{REPO_DIR}/day3")      # gen_metrics 모듈 사용
import gen_metrics as gm

df_full = np.load(f"{ASSET_DIR}/df_preprocessed.npy")
nv_full = np.load(f"{ASSET_DIR}/nv_preprocessed.npy")
print(f"학습용 DF {df_full.shape}, 비교용 NV {nv_full.shape} (값 범위 {df_full.min():.2f}~{df_full.max():.2f})")
assert df_full.shape[1] == IMG_SIZE, f"자산 해상도가 IMG_SIZE({IMG_SIZE})와 다름"

In [ ]:
import torch
import torch.nn as nn

class PatchQuantumGenerator(nn.Module):
    """서브제너레이터 N개가 patch를 나눠 만들고 이어붙여 이미지 한 장을 생성함."""

    def __init__(self, n_generators, n_data_qubits, n_ancillas=1, q_depth=10):
        super().__init__()
        self.n_generators = n_generators
        self.n_qubits = n_data_qubits + n_ancillas
        self.patch_dim = 2 ** n_data_qubits
        self.anc_stride = 2 ** n_ancillas
        q_dev = qml.device("default.qubit", wires=self.n_qubits)
        n_q, depth = self.n_qubits, q_depth

        @qml.qnode(q_dev, interface="torch", diff_method="backprop")
        def circuit(noise, weights):
            for i in range(n_q):
                qml.RY(noise[i], wires=i)
            for layer in range(depth):
                for i in range(n_q):
                    qml.RY(weights[layer, i], wires=i)
                for i in range(n_q - 1):
                    qml.CZ(wires=[i, i + 1])
            return qml.probs(wires=range(n_q))

        self.circuit = circuit
        self.q_params = nn.ParameterList([
            nn.Parameter(torch.rand(q_depth, self.n_qubits) * math.pi) for _ in range(n_generators)
        ])

    def forward(self, batch_size):
        patches = []
        for g in range(self.n_generators):
            w = self.q_params[g]
            rows = [self.circuit(torch.rand(self.n_qubits) * math.pi, w)[::self.anc_stride]
                    for _ in range(batch_size)]
            patches.append(torch.stack(rows))
        return torch.cat(patches, dim=1).float()

    @property
    def output_dim(self):
        return self.n_generators * self.patch_dim


class ClassicalDiscriminator(nn.Module):
    """논문 Table 4 그대로 - 5층 완전연결, ReLU, sigmoid 출력."""

    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


gen = PatchQuantumGenerator(N_GENERATORS, N_DATA_QUBITS, N_ANCILLA, Q_DEPTH)
disc = ClassicalDiscriminator(input_dim=gen.output_dim)
print(f"생성자 출력 차원: {gen.output_dim} (서브제너레이터 {gen.n_generators} x patch {gen.patch_dim})")
print(f"학습 파라미터 수 - 생성자 {sum(p.numel() for p in gen.parameters())}개 / "
      f"판별자 {sum(p.numel() for p in disc.parameters())}개")

In [ ]:
import time

# real을 확률 척도(픽셀 합=1)로 맞춤 - 생성자가 확률을 내므로 같은 척도로 비교해야 함
real_flat = torch.tensor(df_full.reshape(len(df_full), -1), dtype=torch.float32)
real_flat = real_flat / real_flat.sum(dim=1, keepdim=True).clamp(min=1e-6)

bce = nn.BCELoss()
opt_g = torch.optim.Adam(gen.parameters(), lr=0.05)
opt_d = torch.optim.Adam(disc.parameters(), lr=0.001)

STEPS = 400       # 강사 체크포인트는 600스텝(약 22분)으로 학습함. 시간이 부족하면 300으로 줄여도 됨
BATCH = 8
hist_g, hist_d = [], []

t0 = time.time()
for step in range(STEPS):
    batch = real_flat[torch.randint(0, len(real_flat), (BATCH,))]
    real_lbl = torch.ones(BATCH, 1)
    fake_lbl = torch.zeros(BATCH, 1)

    # 판별자
    opt_d.zero_grad()
    loss_d = bce(disc(batch), real_lbl) + bce(disc(gen(BATCH).detach()), fake_lbl)
    loss_d.backward(); opt_d.step()

    # 생성자 (양자 회로 파라미터까지 역전파됨)
    opt_g.zero_grad()
    loss_g = bce(disc(gen(BATCH)), real_lbl)
    loss_g.backward(); opt_g.step()

    hist_g.append(loss_g.item()); hist_d.append(loss_d.item())
    if (step + 1) % 20 == 0:
        el = time.time() - t0
        print(f"step {step+1}/{STEPS}  loss_g={loss_g.item():.4f}  loss_d={loss_d.item():.4f}  "
              f"경과 {el/60:.1f}분  남은 예상 {(STEPS-step-1)*el/(step+1)/60:.1f}분", flush=True)

print(f"\n학습 완료 - 총 {(time.time()-t0)/60:.1f}분")

### 손실곡선 읽는 법

- `loss_d`가 0에 가까워지면 판별자가 일방적으로 이기는 상태임(생성자가 학습 신호를 못 받음).
- 두 손실이 비슷한 수준에서 오르내리면 경쟁이 유지되고 있다는 뜻임.
- GAN은 손실이 단조 감소하지 않음. 손실만 보고 품질을 판단할 수 없어서 다음 섹션의 지표가 필요함.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
gm.plot_loss_curves(hist_g, hist_d, ax=ax)
ax.set_title("QGAN 학습 손실곡선")
plt.tight_layout(); plt.show()

## 6. 생성 + 생성품질 지표 4종

"잘 생성됐는가"를 눈으로만 판단하지 않기 위해 지표 4종을 사용함
(모듈 소스는 저장소 `day3/gen_metrics.py`에서 직접 읽어볼 수 있음).

1. **픽셀 강도 통계·히스토그램** - 실제와 밝기 분포가 비슷한지
2. **클래스 평균 이미지와의 상관계수** - DF의 전형적 형태를 재현했는지(1에 가까우면 좋음)
3. **최근접 실제 이미지 거리** - 0에 가까우면 학습 데이터를 베낀 것(암기)
4. **샘플 간 다양성** - 0에 가까우면 모드붕괴(어떤 노이즈를 넣어도 같은 그림)

지표는 전부 `to_unit_scale`로 0~1로 되돌린 뒤 계산함. 3일차 MediQ-GAN도 **같은 지표**로 평가해
두 모델을 나란히 비교함.

In [ ]:
# 지표는 표본이 적으면 측정마다 흔들림. QGAN은 생성이 싸므로 64장으로 계산함(그림은 앞 4장만 표시)
# 매 호출마다 새 난수를 쓰는 구조라 8~16장으로는 평균 상관계수가 0.05~0.17까지 흔들렸음
N_EVAL = 64
with torch.no_grad():
    fake = gen(N_EVAL).numpy().reshape(N_EVAL, IMG_SIZE, IMG_SIZE)

fake_u = gm.to_unit_scale(fake, "prob")   # 확률 척도 -> 0~1
real_u = gm.to_unit_scale(df_full, "unit")

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j in range(4):
    axes[0, j].imshow(fake_u[j], cmap="gray"); axes[0, j].set_title(f"생성 {j}", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(real_u[j], cmap="gray"); axes[1, j].set_title(f"실제 DF {j}", fontsize=9); axes[1, j].axis("off")
plt.suptitle("QGAN 생성 이미지(위) vs 실제 DF(아래)")
plt.tight_layout(); plt.show()

In [ ]:
# 지표 ①② - 픽셀 분포 + 클래스 평균 상관계수
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
gm.plot_pixel_histogram(real_u, fake_u, ax=axes[0], label_real="실제 DF", label_fake="QGAN 생성")
axes[0].set_title("픽셀 강도 분포")

df_mean = gm.class_mean_image(real_u)
nv_mean = gm.class_mean_image(gm.to_unit_scale(nv_full, "unit"))
corr_df = gm.mean_image_correlation(fake_u, df_mean)
corr_nv = gm.mean_image_correlation(fake_u, nv_mean)
axes[1].bar(["DF 평균 대비", "NV 평균 대비"], [corr_df.mean(), corr_nv.mean()])
axes[1].axhline(0, color="gray", lw=0.8); axes[1].set_ylabel("상관계수")
axes[1].set_title("클래스 평균 이미지와의 상관계수")
plt.tight_layout(); plt.show()

print(f"DF 평균 대비 상관계수: {corr_df.mean():.3f} (샘플별 최댓값 {corr_df.max():.3f})")
print(f"NV 평균 대비 상관계수: {corr_nv.mean():.3f}")
print("DF 쪽이 더 높아야 'DF를 학습했다'고 말할 수 있음. 둘이 비슷하면 클래스 특징을 못 잡은 것임")

In [ ]:
# 지표 ③ - 최근접 실제 이미지(암기 여부 확인)
fig, dist = gm.plot_nearest_neighbors(fake_u, real_u, IMG_SIZE, n_show=4)
plt.show()
print(f"최근접 실제 이미지까지의 L2 거리 - 평균 {dist.mean():.3f}, 최솟값 {dist.min():.3f}")
print("0에 매우 가까우면 학습 데이터를 거의 복사한 것(암기)이므로 증강 가치가 없음")

# 지표 ④ - 샘플 간 다양성
div = gm.sample_diversity(fake_u)
print(f"\n샘플 간 거리 평균 {div['pairwise_mean']:.3f} / 최솟값 {div['pairwise_min']:.3f} / "
      f"픽셀별 표준편차 {div['pixelwise_std']:.3f}")
print("값이 0에 가까우면 모드붕괴임")

In [ ]:
# 4종을 한 표로 - 3일차에서 MediQ-GAN을 같은 표에 추가해 비교함
table = gm.comparison_table({"QGAN(2일차 학습)": (fake, "prob")}, df_full)
print("=== 생성품질 지표 요약 ===")
display(table) if "display" in dir() else print(table)

## 7. 정리 - Drive에 저장

오늘 학습한 **QGAN 체크포인트**와 직접 전처리한 샘플을 Google Drive에 저장함.
3일차는 강사가 전체 데이터로 준비한 자산을 저장소에서 받아 쓰지만, 본인이 학습시킨 모델로도 이어서 해볼 수 있음.
원본 데이터(약 6GB)는 저장하지 않음.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/동아대_QML강의"
os.makedirs(save_dir, exist_ok=True)

torch.save({
    "generator_state_dict": gen.state_dict(),
    "config": {"n_generators": N_GENERATORS, "n_data_qubits": N_DATA_QUBITS,
               "n_ancillas": N_ANCILLA, "q_depth": Q_DEPTH, "img_size": IMG_SIZE,
               "target_class": "df", "scale": "prob", "steps": STEPS},
    "history": {"loss_g": hist_g, "loss_d": hist_d},
}, f"{save_dir}/qgan_my_{IMG_SIZE}px.pt")
np.save(f"{save_dir}/df_demo_{IMG_SIZE}px.npy", df_demo)
print(f"저장 완료: {save_dir}")
print("3일차는 저장소 자산(day3/assets)으로 진행하며, 본인 체크포인트로 해보고 싶으면 이 파일을 불러와 쓸 수 있음")

---
## 2일차 마무리 체크리스트
- [ ] Colab 환경(PennyLane, kagglehub 등) 정상 설치 확인
- [ ] kagglehub로 HAM10000 다운로드 완료
- [ ] HAM10000 EDA 완료 - 클래스 불균형 확인
- [ ] DF/NV 세그멘테이션 전처리 직접 실행
- [ ] 서브제너레이터 회로(데이터 6 + 보조 1큐빗) 구성 + 출력 확인
- [ ] QGAN 학습 완료 - 손실곡선 확인
- [ ] 생성 이미지 + 지표 4종 확인
- [ ] 체크포인트 Google Drive에 저장

**질문이 있으면 지금 제시할 것 - 3일차는 오늘 학습한 QGAN을 MediQ-GAN과 비교하는 것부터 시작함.**